# 00 — Visão geral: as três fases de um projeto de Engenharia de Dados

> **Objetivo desta aula.** Construir, do zero, um pipeline de dados completo e
> funcional — e entender *por que* cada peça existe.

Ao final da sequência de notebooks você vai ter, rodando na sua máquina:

1. uma **ingestão** que consome uma API pública de verdade, com paginação,
   controle de vazão e reprocessamento seguro;
2. um conjunto de **validações de qualidade** com Great Expectations, que
   impede dado ruim de avançar no pipeline;
3. um **processamento em camadas** que transforma JSON aninhado em tabelas
   analíticas;
4. um **aplicativo analítico** em Streamlit que qualquer pessoa consegue usar.

O domínio escolhido é a **Fórmula 1**: dados públicos, ricos, com regras de
negócio interessantes e — importante para o ensino — com resultados que você
consegue conferir contra a realidade.

## 1. O problema que a Engenharia de Dados resolve

Um dado bruto raramente serve para decisão. Ele chega:

* **aninhado** — JSON com listas dentro de listas;
* **sem tipo** — tudo é texto, inclusive número e data;
* **paginado** — 480 registros chegam em 5 requisições;
* **inconsistente** — o mesmo conceito muda de nome ao longo do tempo;
* **sem contexto** — `"max_verstappen"` não diz quantos pontos ele fez.

O trabalho da engenharia de dados é levar esse dado do estado "cru e
inutilizável" até o estado "confiável e pronto para responder perguntas" —
e fazer isso de forma **repetível**, **auditável** e **automatizável**.

Este projeto organiza esse caminho em três fases.

## 2. As três fases

```
   FASE 1: INGESTÃO            FASE 2: PROCESSAMENTO         FASE 3: ANALYTICS
 ┌───────────────────┐      ┌──────────────────────────┐   ┌──────────────────┐
 │  API pública F1   │      │  bronze → silver → gold  │   │  App Streamlit   │
 │  (Jolpica/Ergast) │      │                          │   │                  │
 └─────────┬─────────┘      └────────────┬─────────────┘   └────────▲─────────┘
           │                             │                          │
           ▼                             ▼                          │
     data/raw/*.json  ──────────►  data/bronze  ──► data/silver ──► data/gold
     (JSON cru,                   (tabular,        (tipado,       (agregado,
      particionado,                fiel à           modelo         pronto para
      com manifesto)               fonte)           estrela)       consumo)
           │                             │                          │
           └────────► ✅ GREAT EXPECTATIONS valida cada passagem ◄───┘
                    (contrato da fonte, semântica, reconciliação)
```

| Fase | Notebook | Ferramenta | Pergunta que responde |
|---|---|---|---|
| 1. Ingestão | `01_ingestao.ipynb` | Python + `requests` | Como trago o dado para casa sem perder nada e sem derrubar a fonte? |
| 1.5 Qualidade | `02_qualidade_great_expectations.ipynb` | Great Expectations | Posso confiar no que acabei de trazer? |
| 2. Processamento | `03_processamento_agregacoes.ipynb` | Python + pandas | Como transformo isso em algo que responde perguntas? |
| 3. Analytics | `04_analytics.ipynb` + app | Streamlit + Plotly | Como coloco isso na mão de quem decide? |

## 3. Por que camadas? (arquitetura *medallion*)

Poderíamos ir da API direto ao gráfico. Quase todo mundo tenta isso primeiro —
e quase todo mundo se arrepende. Separar em camadas resolve quatro problemas
concretos:

| Camada | Regra | O que ganho com isso |
|---|---|---|
| **raw** | grave o payload **exatamente** como veio | achou um bug 3 meses depois? reprocessa sem chamar a API de novo |
| **bronze** | achate para tabela, **sem** converter tipo nem filtrar | dá para comparar linha a linha com o JSON: é auditável |
| **silver** | tipe, limpe, aplique regra de negócio, modele | uma única definição de "abandono" para o projeto inteiro |
| **gold** | agregue por pergunta de negócio | o app não faz JOIN nem regra: ele só desenha |

> **A regra que sustenta tudo:** cada camada tem **uma** responsabilidade. Quando
> uma transformação está "no lugar errado", o sintoma aparece meses depois, como
> duas telas mostrando números diferentes para a mesma pergunta.

## 4. A fonte de dados

Usamos a **[Jolpica-F1 API](https://github.com/jolpica/jolpica-f1)** — sucessora
mantida pela comunidade da clássica Ergast API (desativada no fim de 2024).
É pública, não exige chave e mantém o mesmo contrato da Ergast.

```
https://api.jolpi.ca/ergast/f1/{temporada}/results.json?limit=100&offset=0
```

**Limites que respeitamos** (e que você vai encontrar em qualquer API real):

* ~4 requisições por segundo, 500 por hora sem autenticação;
* no máximo **100 registros por página** — daí a necessidade de paginar;
* alguns recursos só existem por corrida, não por temporada.

> ⚠️ **Etiqueta de ingestão.** Uma API pública gratuita é mantida por voluntários.
> Espaçar requisições, identificar-se no `User-Agent` e não rebaixar o mesmo dado
> mil vezes não é detalhe técnico: é o que mantém a fonte no ar para todo mundo.

## 5. Organização do repositório

```
Estudo-MDS/
├── notebooks/          ← a aula: 00 a 04, na ordem
├── src/f1_pipeline/    ← o código de produção (o que os notebooks chamam)
│   ├── ingestion/      ← fase 1: cliente HTTP, endpoints, jobs
│   ├── quality/        ← Great Expectations: contexto e suites de regras
│   ├── processing/     ← fase 2: bronze.py, silver.py, gold.py
│   └── utils/          ← logging e I/O
├── app/                ← fase 3: aplicativo Streamlit (multipágina)
├── data/               ← raw / bronze / silver / gold  (não vai para o Git)
├── docs/data_docs/     ← relatório HTML do Great Expectations
├── reports/            ← relatório de qualidade em JSON (lido pelo app)
├── scripts/            ← run_pipeline.py: o pipeline inteiro em um comando
└── tests/              ← pytest: testa a LÓGICA (o GX testa o DADO)
```

**Por que o código não mora dentro do notebook?** Porque notebook é ótimo para
*explicar* e péssimo para *reusar*. A regra que seguimos aqui — e que vale para
qualquer projeto sério — é: a lógica vive em `src/`, testada e importável; o
notebook conta a história, chama a lógica e mostra o resultado.

## 6. Preparando o ambiente

No terminal, na raiz do projeto:

```bash
python -m venv .venv
# Windows:
.venv\Scripts\activate
# Linux/macOS:
source .venv/bin/activate

pip install -r requirements.txt
```

Depois rode a célula abaixo para conferir se está tudo no lugar.

In [ ]:
# --- Preparação do ambiente (rode esta célula primeiro) --------------------
import sys
from pathlib import Path

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ / "src"))

import pandas as pd

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)

print("Raiz do projeto:", RAIZ)
print("Python:", sys.version.split()[0])


In [ ]:
# Conferindo as dependências das três fases
import importlib

dependencias = {
    "requests": "Fase 1 — cliente HTTP",
    "tenacity": "Fase 1 — retry com backoff",
    "great_expectations": "Fase 1.5 — qualidade de dados",
    "pandas": "Fase 2 — processamento",
    "pyarrow": "Fase 2 — escrita em Parquet",
    "streamlit": "Fase 3 — aplicativo analítico",
    "plotly": "Fase 3 — gráficos interativos",
}

for pacote, papel in dependencias.items():
    try:
        modulo = importlib.import_module(pacote)
        versao = getattr(modulo, "__version__", "?")
        print(f"OK   {pacote:<20} {versao:<12} {papel}")
    except ImportError:
        print(f"FALTA {pacote:<19} {'':<12} {papel}  ->  pip install {pacote}")


In [ ]:
# O projeto inteiro é configurado em um único lugar: src/f1_pipeline/config.py
from f1_pipeline import config

print("Temporadas que serão carregadas :", list(config.SEASONS))
print("URL base da API                 :", config.API_BASE_URL)
print("Intervalo mínimo entre chamadas :", config.API_MIN_INTERVAL_SECONDS, "s")
print()
for camada in config.ALL_DATA_DIRS:
    existe = "existe" if camada.exists() else "ainda não existe"
    print(f"{camada.name:<8} -> {camada}  ({existe})")


### Primeiro contato com a fonte

Antes de escrever qualquer pipeline, o passo zero é sempre o mesmo: **olhar o
dado**. Uma requisição, um payload, um olhar humano.

In [ ]:
import json
import requests

url = f"{config.API_BASE_URL}/2024/1/results.json"
resposta = requests.get(url, params={"limit": 3}, timeout=30, headers={"User-Agent": config.API_USER_AGENT})
resposta.raise_for_status()
payload = resposta.json()

print("HTTP", resposta.status_code, "|", len(resposta.content), "bytes")
print("Estrutura de mais alto nível:", list(payload.keys()))
print("Dentro de MRData:", list(payload["MRData"].keys()))


In [ ]:
# O primeiro resultado da primeira corrida de 2024, cru:
primeiro = payload["MRData"]["RaceTable"]["Races"][0]["Results"][0]
print(json.dumps(primeiro, indent=2, ensure_ascii=False))


Repare em três coisas que definem todo o trabalho a seguir:

1. **Tudo é texto.** `"points": "26"`, `"grid": "1"` — não dá para somar nem
   ordenar antes de converter.
2. **Está aninhado em três níveis.** `Races[] → Results[] → Driver{}`. Uma tabela
   é plana; alguém precisa achatar isso.
3. **O contexto está longe do fato.** O nome do GP está dois níveis acima do
   resultado do piloto. Ao achatar, temos de trazê-lo junto.

Nada disso é defeito da API — é a natureza de um payload de API. É exatamente
por isso que a fase de processamento existe.

## 7. Roteiro dos notebooks

| Notebook | O que você vai fazer |
|---|---|
| **01 — Ingestão** | Escrever/entender um cliente com retry, paginação e rate limit. Gravar a camada raw particionada, com manifesto e hash. Discutir idempotência. |
| **02 — Qualidade** | Achatar para bronze e validar o **contrato da fonte** com Great Expectations. Quebrar o dado de propósito para ver a validação reprovar. Gerar os Data Docs. |
| **03 — Processamento** | bronze → silver (tipagem, regras de negócio, modelo estrela) → gold (agregações). Validar de novo, agora a semântica e a reconciliação. |
| **04 — Analytics** | Explorar a camada gold, montar os gráficos e subir o app Streamlit. |

> **Atenção à ordem.** Cada notebook consome o que o anterior produziu. Se pular
> o 01, o 02 não encontra dado — e isso é proposital: dependência entre etapas é
> a realidade de qualquer pipeline.

Se em algum momento você quiser rodar tudo de uma vez (por exemplo, para
recomeçar do zero):

```bash
python scripts/run_pipeline.py --seasons 2021-2024
```

---

**Próximo:** [`01_ingestao.ipynb`](01_ingestao.ipynb)